# H3: NORMALIZED SGD vs MUON -- Is the Polar Factor Necessary, or Is Scale Removal Sufficient?

## The Central Scientific Question

The Muon optimizer replaces gradient updates with the **polar factor** $U V^T$ from the SVD of the
gradient $G = U \Sigma V^T$. This discards all singular value information, replacing the gradient's
spectrum with a flat spectrum of all ones. But *why* does this work? Two competing hypotheses:

**Hypothesis A -- Scale Collapse Sufficiency:** The benefit comes entirely from **removing gradient
magnitude** (scale collapse). Any normalization that prevents the optimizer from being dominated by
gradient scale should produce the same "Muon paradox" (high weight diversity, low functional diversity)
and similar convergence. The polar factor's orthogonal structure is incidental.

**Hypothesis B -- Polar Factor Necessity:** The orthogonal projection $U V^T$ does something
**qualitatively different** from mere normalization. It doesn't just remove scale -- it redistributes
update energy equally across all singular directions. Simple normalization (Frobenius, spectral, or
sign) cannot replicate this, and the paradox strength and/or convergence quality will differ.

## Why This Matters for the RG-Gauge-Fixing Theory

In the renormalization group (RG) framework, Muon's polar factor is interpreted as a **gauge-fixing
operation** that projects gradient updates onto the orthogonal manifold $O(n)$. If simple normalization
suffices, then the "gauge" being fixed is merely gradient scale -- a trivial gauge. If the polar factor
is essential, then the gauge structure is richer: it involves the full rotational degrees of freedom of
weight matrices, and Muon is performing a genuine geometric projection onto a curved manifold.

## Experimental Design

We compare five optimizers, all using momentum = 0.9:

| Optimizer | Update Rule | What It Removes |
|-----------|------------|-----------------|
| **SGD** | $W \leftarrow W - \eta \cdot v$ | Nothing (baseline) |
| **Muon** | $v \leftarrow \beta v + \text{NS}(G)$; orthogonalize then momentum | All singular values (polar factor) |
| **Normalized SGD** | $v \leftarrow \beta v + G$; step $= v / \|v\|_F$ | Overall Frobenius scale |
| **Spectral SGD** | $v \leftarrow \beta v + G$; step $= v / \|v\|_2$ | Largest singular value only |
| **Sign SGD** | $v \leftarrow \beta v + G$; step $= \text{sign}(v)$ | All magnitudes (element-wise) |

Each optimizer is tested on **4-layer deep linear** and **4-layer ReLU** networks (32x32 weight
matrices) with 20 independent random initializations per optimizer. We measure:

- **Loss convergence** (speed and final value)
- **Weight diversity** (pairwise Frobenius distance between final weights from different seeds)
- **Functional diversity** (pairwise distance between network outputs on held-out test inputs)
- **Paradox ratio** = Func_Div / Weight_Div (lower = stronger paradox: weights differ but functions agree)
- **Per-layer condition numbers** at convergence

The key discriminator: if Normalized SGD matches Muon on *both* paradox strength and convergence,
Hypothesis A wins. If Muon uniquely excels, Hypothesis B wins. A middle ground -- normalization
creates the paradox but Muon converges better -- suggests a **two-mechanism story**.

In [ ]:
"""
H3: NORMALIZED SGD vs MUON — Is the Polar Factor Necessary?
=============================================================

THE QUESTION:
  If the Muon Paradox comes from discarding gradient MAGNITUDE (scale collapse),
  then simple normalized SGD (step = G/||G||_F) should show similar paradox strength
  AND convergence benefit. The polar factor would be unnecessary — just normalizing
  the gradient norm is enough.

OPTIMIZERS COMPARED (all with momentum=0.9):
  (a) SGD — baseline
  (b) Muon — ortho of momentum via Newton-Schulz (UV^T)
  (c) Normalized SGD — step = lr * M / ||M||_F  (Frobenius normalization)
  (d) Spectral-normalized SGD — step = lr * M / ||M||_op  (spectral norm)
  (e) Sign SGD — step = lr * sign(M)  (element-wise sign)

KEY TESTS:
  T1: Does normalized SGD create the paradox?
  T2: Does normalized SGD match Muon's convergence speed?
  T3: Does Muon beat normalized SGD on LOSS?
  T4: Which normalization creates the strongest paradox?

Setup: 4-layer nets (32x32), quadratic loss, 20 independent runs.
Architectures: deep linear + ReLU.
"""

In [ ]:
import numpy as np
import os
import time

In [ ]:
np.random.seed(42)

## Section 1: Experimental Configuration

The hyperparameters below define the controlled experimental environment. Key design choices:

- **DIM = 32**: Large enough for non-trivial spectral structure (32 singular values per weight matrix)
  but small enough for exact SVD computation and 20-run statistical sampling.
- **NUM_LAYERS = 4**: Deep enough to expose the gauge symmetry problem (deeper networks have more
  gauge degrees of freedom in how the product $W_4 W_3 W_2 W_1$ is factored).
- **NUM_STEPS = 500**: Sufficient for all optimizers to reach a plateau or near-convergence.
- **MOMENTUM = 0.9**: Standard heavy-ball momentum, held constant across all optimizers to isolate
  the effect of the gradient transformation (normalization vs. polar projection).
- **NS_ITERS = 5**: Newton-Schulz iterations for Muon's polar decomposition. Five iterations give
  machine-precision convergence for 32x32 matrices.
- **NUM_INDEPENDENT_RUNS = 20**: Each optimizer is run from 20 different random initializations.
  The *diversity* across these 20 endpoints is our primary observable -- not just the loss.

In [ ]:
DIM = 32
NUM_LAYERS = 4
NUM_STEPS = 500
BATCH_SIZE = 64
MOMENTUM = 0.9
NS_ITERS = 5
NUM_INDEPENDENT_RUNS = 20
NUM_TEST_INPUTS = 50

print("=" * 90)
print("EXPERIMENTAL CONFIGURATION")
print("=" * 90)
print(f"  Weight matrix dimension:      {DIM} x {DIM}  ({DIM**2} parameters per layer)")
print(f"  Number of layers:             {NUM_LAYERS}  (total parameters = {NUM_LAYERS * DIM**2})")
print(f"  Training steps:               {NUM_STEPS}")
print(f"  Batch size:                   {BATCH_SIZE}")
print(f"  Momentum coefficient:         {MOMENTUM}")
print(f"  Newton-Schulz iterations:     {NS_ITERS}")
print(f"  Independent random seeds:     {NUM_INDEPENDENT_RUNS}")
print(f"  Test inputs for func diversity: {NUM_TEST_INPUTS}")
print(f"")
print(f"  Gauge degrees of freedom in a {NUM_LAYERS}-layer linear net:")
print(f"    Between each pair of adjacent layers, any invertible G satisfying")
print(f"    W_{{i+1}} G^{{-1}} G W_i = W_{{i+1}} W_i is a gauge transformation.")
print(f"    There are {NUM_LAYERS - 1} such inter-layer gauges, each living in GL({DIM}),")
print(f"    giving {(NUM_LAYERS - 1) * DIM**2} gauge degrees of freedom out of")
print(f"    {NUM_LAYERS * DIM**2} total parameters.")
print(f"    Fraction gauge: {(NUM_LAYERS - 1) / NUM_LAYERS:.1%}")
print("=" * 90)

In [ ]:
# Fixed target and data
W_target = np.random.randn(DIM, DIM) * 0.5
X_data = np.random.randn(DIM, BATCH_SIZE) * 0.3
X_test = np.random.randn(DIM, NUM_TEST_INPUTS) * 0.3

# Diagnostic: characterize the target and data
_target_svs = np.linalg.svd(W_target, compute_uv=False)
_target_cond = _target_svs[0] / _target_svs[-1] if _target_svs[-1] > 1e-15 else np.inf
_target_rank = np.sum(_target_svs > 1e-10)
print("TARGET MATRIX PROPERTIES:")
print(f"  W_target shape:        {W_target.shape}")
print(f"  W_target Frobenius norm: {np.linalg.norm(W_target, 'fro'):.4f}")
print(f"  Singular value range:  [{_target_svs[-1]:.4f}, {_target_svs[0]:.4f}]")
print(f"  Condition number:      {_target_cond:.2f}")
print(f"  Effective rank:        {_target_rank}/{DIM}")
print(f"  Spectral norm:         {_target_svs[0]:.4f}")
print(f"")
print("DATA PROPERTIES:")
print(f"  X_data shape:          {X_data.shape}  (training)")
print(f"  X_test shape:          {X_test.shape}  (held-out for functional diversity)")
print(f"  X_data Frobenius norm: {np.linalg.norm(X_data, 'fro'):.4f}")
print(f"  X_test Frobenius norm: {np.linalg.norm(X_test, 'fro'):.4f}")
print(f"")
print("  The target W_target is a full-rank random matrix scaled by 0.5.")
print(f"  Its condition number ({_target_cond:.1f}) means the learning problem is")
print(f"  moderately anisotropic -- not trivially isotropic, not pathologically ill-conditioned.")

In [ ]:
SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))

In [ ]:
OPTIMIZER_NAMES = ['sgd', 'muon', 'norm_sgd', 'spectral_sgd', 'sign_sgd']
OPTIMIZER_LABELS = {
    'sgd': 'SGD',
    'muon': 'Muon (UV^T)',
    'norm_sgd': 'Normalized SGD (Frob)',
    'spectral_sgd': 'Spectral-Norm SGD',
    'sign_sgd': 'Sign SGD',
}
OPTIMIZER_COLORS = {
    'sgd': '#4477AA',
    'muon': '#CC3311',
    'norm_sgd': '#228B22',
    'spectral_sgd': '#9933CC',
    'sign_sgd': '#FF8800',
}

print("OPTIMIZER COMPARISON MATRIX:")
print(f"  {'Optimizer':<30} {'Normalization Type':<30} {'What Gets Removed'}")
print(f"  {'-'*90}")
print(f"  {'SGD':<30} {'None':<30} {'Nothing -- raw gradient with momentum'}")
print(f"  {'Muon (UV^T)':<30} {'Polar decomposition':<30} {'ALL singular values -> flat spectrum'}")
print(f"  {'Normalized SGD (Frob)':<30} {'Frobenius norm':<30} {'Overall scale only'}")
print(f"  {'Spectral-Norm SGD':<30} {'Spectral (operator) norm':<30} {'Largest SV only'}")
print(f"  {'Sign SGD':<30} {'Element-wise sign':<30} {'All element magnitudes'}")
print(f"")
print("  CRITICAL DISTINCTION:")
print("    Muon: sigma_i -> 1 for ALL i  (complete spectral flattening)")
print("    Norm SGD: sigma_i -> sigma_i / sqrt(sum sigma_j^2)  (rescaled, NOT flattened)")
print("    Spectral SGD: sigma_i -> sigma_i / sigma_1  (rescaled by max, NOT flattened)")
print("    Sign SGD: element-wise sign  (destroys correlation structure entirely)")

## Section 2: Network Definitions

Two architectures are tested to separate gauge-specific effects from nonlinearity effects:

**Deep Linear Network** ($f(X) = W_4 W_3 W_2 W_1 X$): The product $W_4 W_3 W_2 W_1$ has an
exact gauge symmetry -- for any invertible $G_i$, replacing $W_{i+1} \to W_{i+1} G_i^{-1}$ and
$W_i \to G_i W_i$ leaves the network function unchanged. This creates a continuous manifold of
weight-space solutions that all compute the same function. The "Muon paradox" is the observation
that Muon explores this manifold more aggressively (higher weight diversity) while still converging
to the same loss and function (low functional diversity).

**ReLU Network** ($f(X) = W_4 \cdot \text{ReLU}(W_3 \cdot \text{ReLU}(W_2 \cdot \text{ReLU}(W_1 X)))$):
ReLU activations break the continuous gauge symmetry down to a discrete (positive-scaling) symmetry.
This tests whether the paradox persists when the gauge group is smaller.

### Weight Initialization

Weights are initialized as $W = I + 0.1 \cdot \text{randn}$, starting near the identity. This ensures
the initial network is close to a well-conditioned regime and the product $W_4 W_3 W_2 W_1 \approx I$.

In [ ]:
def init_weights(num_layers, seed=42):
    rng = np.random.RandomState(seed)
    weights = []
    for _ in range(num_layers):
        W = np.eye(DIM) + rng.randn(DIM, DIM) * 0.1
        weights.append(W.copy())
    return weights

# Diagnostic: characterize a sample initialization
_sample_weights = init_weights(NUM_LAYERS, seed=42)
print("INITIALIZATION DIAGNOSTICS (seed=42):")
for _li, _W in enumerate(_sample_weights):
    _svs = np.linalg.svd(_W, compute_uv=False)
    _cond = _svs[0] / _svs[-1] if _svs[-1] > 1e-15 else np.inf
    print(f"  Layer {_li}: ||W||_F={np.linalg.norm(_W, 'fro'):.3f}  "
          f"sigma_range=[{_svs[-1]:.3f}, {_svs[0]:.3f}]  "
          f"cond={_cond:.2f}  "
          f"||W - I||_F={np.linalg.norm(_W - np.eye(DIM), 'fro'):.3f}")
_product = _sample_weights[0]
for _W in _sample_weights[1:]:
    _product = _W @ _product
_prod_svs = np.linalg.svd(_product, compute_uv=False)
_prod_cond = _prod_svs[0] / _prod_svs[-1] if _prod_svs[-1] > 1e-15 else np.inf
print(f"  Product W4*W3*W2*W1: ||prod||_F={np.linalg.norm(_product, 'fro'):.3f}  "
      f"cond={_prod_cond:.2f}  "
      f"||prod - I||_F={np.linalg.norm(_product - np.eye(DIM), 'fro'):.3f}")
print(f"  Initial product is close to identity (perturbation ~{np.linalg.norm(_product - np.eye(DIM), 'fro'):.2f})")

### Deep Linear Network

Forward pass: $f(X) = W_4 W_3 W_2 W_1 X$. Loss: $\mathcal{L} = \frac{1}{2N} \sum_j \| f(x_j) - W^* x_j \|^2$.

Gradients are computed via standard backpropagation through the chain of matrix multiplications.
The key property: for a deep linear network, the gradient $\partial \mathcal{L} / \partial W_i$ depends
on all other weight matrices through the chain rule, creating coupled dynamics where each layer's
gradient is shaped by the spectral properties of all other layers.

In [ ]:
def forward_linear(weights, X):
    out = X.copy()
    for W in weights:
        out = W @ out
    return out

In [ ]:
def compute_loss_linear(weights, X, target):
    pred = forward_linear(weights, X)
    target_out = target @ X
    diff = pred - target_out
    return 0.5 * np.mean(np.sum(diff ** 2, axis=0))

In [ ]:
def compute_gradients_linear(weights, X, target):
    num_layers = len(weights)
    N = X.shape[1]
    activations = [X.copy()]
    out = X.copy()
    for W in weights:
        out = W @ out
        activations.append(out.copy())
    target_out = target @ X
    delta = (activations[-1] - target_out) / N
    grads = []
    for i in range(num_layers - 1, -1, -1):
        G = delta @ activations[i].T
        grads.insert(0, G)
        if i > 0:
            delta = weights[i].T @ delta
    return grads

### ReLU Network

Forward pass: $f(X) = W_4 \cdot \text{ReLU}(W_3 \cdot \text{ReLU}(W_2 \cdot \text{ReLU}(W_1 X)))$.

ReLU activations between layers break the continuous GL(n) gauge symmetry. In a deep linear net,
$W_{i+1} G^{-1} \cdot G W_i = W_{i+1} W_i$ for any invertible $G$. With ReLU, only **positive
diagonal** rescalings $G = \text{diag}(\alpha_1, ..., \alpha_n)$ with $\alpha_j > 0$ preserve the
network function (since ReLU commutes with positive scaling). This reduces the gauge group from
$GL(n)$ (with $n^2$ parameters) to $\mathbb{R}^n_+$ (with $n$ parameters per inter-layer boundary).

If the paradox is purely a gauge phenomenon, we expect it to be **weaker** in ReLU networks
(smaller gauge group = fewer redundant directions in weight space).

In [ ]:
def forward_relu(weights, X):
    out = X.copy()
    for idx, W in enumerate(weights):
        out = W @ out
        if idx < len(weights) - 1:
            out = np.maximum(0, out)
    return out

In [ ]:
def compute_loss_relu(weights, X, target):
    pred = forward_relu(weights, X)
    target_out = target @ X
    diff = pred - target_out
    return 0.5 * np.mean(np.sum(diff ** 2, axis=0))

In [ ]:
def compute_gradients_relu(weights, X, target):
    num_layers = len(weights)
    N = X.shape[1]
    pre_activations = []
    post_activations = [X.copy()]
    out = X.copy()
    for idx, W in enumerate(weights):
        pre = W @ out
        pre_activations.append(pre.copy())
        if idx < num_layers - 1:
            out = np.maximum(0, pre)
        else:
            out = pre
        post_activations.append(out.copy())
    target_out = target @ X
    delta = (post_activations[-1] - target_out) / N
    grads = []
    for i in range(num_layers - 1, -1, -1):
        G = delta @ post_activations[i].T
        grads.insert(0, G)
        if i > 0:
            delta = weights[i].T @ delta
            delta = delta * (pre_activations[i - 1] > 0).astype(float)
    return grads

## Section 3: Optimizer Implementations

### Newton-Schulz Orthogonalization (Muon's Core)

The Newton-Schulz iteration computes the polar factor $U V^T$ of a matrix $G$ without performing
an explicit SVD. Starting from $X_0 = G / \|G\|_F$, it iterates:

$$X_{k+1} = \frac{3}{2} X_k - \frac{1}{2} X_k (X_k^T X_k)$$

This converges cubically to the closest orthogonal matrix to $G$ in Frobenius norm. After 5 iterations
on a 32x32 matrix, the result satisfies $X^T X \approx I$ to machine precision.

**What this does spectrally:** If $G = U \Sigma V^T$, then Newton-Schulz converges to $U V^T$,
which has singular values all equal to 1. The *directions* ($U$, $V$) are preserved but the
*magnitudes* ($\Sigma$) are completely erased and replaced with uniform unit values.

### The Five Optimizer Update Rules

All optimizers use the same momentum coefficient $\beta = 0.9$. The key difference is what
happens to the momentum buffer *after* accumulation and *before* the weight update:

- **SGD**: No transformation. The raw momentum buffer drives the update.
- **Muon**: Orthogonalize each gradient *before* adding to momentum. The momentum buffer
  thus accumulates orthogonalized gradients.
- **Norm SGD**: Accumulate raw gradients into momentum, then divide by Frobenius norm.
- **Spectral SGD**: Accumulate raw gradients into momentum, then divide by spectral norm.
- **Sign SGD**: Accumulate raw gradients into momentum, then take element-wise sign.

In [ ]:
def newton_schulz_orthogonalize(G, num_iters=NS_ITERS):
    norm = np.linalg.norm(G, ord='fro')
    if norm < 1e-12:
        return G
    X = G / norm
    for _ in range(num_iters):
        A = X.T @ X
        X = 1.5 * X - 0.5 * X @ A
    return X

# Verify Newton-Schulz convergence on a sample gradient
_test_grad = np.random.randn(DIM, DIM)
_test_svs_before = np.linalg.svd(_test_grad, compute_uv=False)
_test_ortho = newton_schulz_orthogonalize(_test_grad)
_test_svs_after = np.linalg.svd(_test_ortho, compute_uv=False)
_ortho_error = np.linalg.norm(_test_ortho.T @ _test_ortho - np.eye(DIM), 'fro')
print("NEWTON-SCHULZ VERIFICATION (on random 32x32 gradient):")
print(f"  Input singular values:   min={_test_svs_before[-1]:.4f}  max={_test_svs_before[0]:.4f}  "
      f"ratio={_test_svs_before[0]/_test_svs_before[-1]:.2f}")
print(f"  Output singular values:  min={_test_svs_after[-1]:.6f}  max={_test_svs_after[0]:.6f}  "
      f"ratio={_test_svs_after[0]/_test_svs_after[-1]:.6f}")
print(f"  Orthogonality error ||X^T X - I||_F = {_ortho_error:.2e}")
print(f"  After {NS_ITERS} iterations, all singular values are {'~1.0 (GOOD)' if abs(_test_svs_after[0] - 1.0) < 1e-6 else 'NOT converged (BAD)'}")
print(f"  This confirms complete spectral flattening: the polar factor erases ALL scale information.")

In [ ]:
def optimizer_step(weights, velocities, grads, lr, method):
    """Unified optimizer step for all 5 methods."""
    for i in range(len(weights)):
        if method == 'sgd':
            velocities[i] = MOMENTUM * velocities[i] + grads[i]
            weights[i] = weights[i] - lr * velocities[i]

        elif method == 'muon':
            # Muon: orthogonalize the gradient, then apply momentum
            ortho_grad = newton_schulz_orthogonalize(grads[i])
            velocities[i] = MOMENTUM * velocities[i] + ortho_grad
            weights[i] = weights[i] - lr * velocities[i]

        elif method == 'norm_sgd':
            # Standard momentum, then normalize momentum to unit Frobenius
            velocities[i] = MOMENTUM * velocities[i] + grads[i]
            v_norm = np.linalg.norm(velocities[i], 'fro')
            if v_norm > 1e-12:
                step = velocities[i] / v_norm
            else:
                step = velocities[i]
            weights[i] = weights[i] - lr * step

        elif method == 'spectral_sgd':
            # Standard momentum, then normalize to unit spectral norm
            velocities[i] = MOMENTUM * velocities[i] + grads[i]
            spec_norm = np.linalg.norm(velocities[i], ord=2)
            if spec_norm > 1e-12:
                step = velocities[i] / spec_norm
            else:
                step = velocities[i]
            weights[i] = weights[i] - lr * step

        elif method == 'sign_sgd':
            # Standard momentum, then take element-wise sign
            velocities[i] = MOMENTUM * velocities[i] + grads[i]
            step = np.sign(velocities[i])
            weights[i] = weights[i] - lr * step

    return weights, velocities

# Diagnostic: demonstrate what each optimizer does to the SAME gradient
print("OPTIMIZER SPECTRAL SIGNATURE COMPARISON:")
print("  (Applying each optimizer's transformation to the same random gradient)")
_demo_G = np.random.randn(DIM, DIM) * 0.1
_demo_svs_raw = np.linalg.svd(_demo_G, compute_uv=False)
print(f"\n  Raw gradient: sigma_range=[{_demo_svs_raw[-1]:.4f}, {_demo_svs_raw[0]:.4f}]  "
      f"cond={_demo_svs_raw[0]/_demo_svs_raw[-1]:.2f}  "
      f"||G||_F={np.linalg.norm(_demo_G, 'fro'):.4f}")

# Muon (polar factor)
_demo_ortho = newton_schulz_orthogonalize(_demo_G)
_demo_svs_muon = np.linalg.svd(_demo_ortho, compute_uv=False)
print(f"  Muon (UV^T):       sigma_range=[{_demo_svs_muon[-1]:.4f}, {_demo_svs_muon[0]:.4f}]  "
      f"cond={_demo_svs_muon[0]/_demo_svs_muon[-1]:.4f}  "
      f"||step||_F={np.linalg.norm(_demo_ortho, 'fro'):.4f}")

# Frobenius normalization
_demo_frob = _demo_G / np.linalg.norm(_demo_G, 'fro')
_demo_svs_frob = np.linalg.svd(_demo_frob, compute_uv=False)
print(f"  Norm SGD (Frob):   sigma_range=[{_demo_svs_frob[-1]:.4f}, {_demo_svs_frob[0]:.4f}]  "
      f"cond={_demo_svs_frob[0]/_demo_svs_frob[-1]:.2f}  "
      f"||step||_F={np.linalg.norm(_demo_frob, 'fro'):.4f}")

# Spectral normalization
_demo_spec = _demo_G / np.linalg.norm(_demo_G, ord=2)
_demo_svs_spec = np.linalg.svd(_demo_spec, compute_uv=False)
print(f"  Spectral SGD:      sigma_range=[{_demo_svs_spec[-1]:.4f}, {_demo_svs_spec[0]:.4f}]  "
      f"cond={_demo_svs_spec[0]/_demo_svs_spec[-1]:.2f}  "
      f"||step||_F={np.linalg.norm(_demo_spec, 'fro'):.4f}")

# Sign
_demo_sign = np.sign(_demo_G)
_demo_svs_sign = np.linalg.svd(_demo_sign, compute_uv=False)
print(f"  Sign SGD:          sigma_range=[{_demo_svs_sign[-1]:.4f}, {_demo_svs_sign[0]:.4f}]  "
      f"cond={_demo_svs_sign[0]/_demo_svs_sign[-1]:.2f}  "
      f"||step||_F={np.linalg.norm(_demo_sign, 'fro'):.4f}")

print(f"\n  KEY OBSERVATION:")
print(f"    Muon condition number: {_demo_svs_muon[0]/_demo_svs_muon[-1]:.4f} (perfectly 1.0 = flat spectrum)")
print(f"    Frob condition number: {_demo_svs_frob[0]/_demo_svs_frob[-1]:.2f} (same as raw -- only scale changes)")
print(f"    Spectral cond number:  {_demo_svs_spec[0]/_demo_svs_spec[-1]:.2f} (same as raw -- only scale changes)")
print(f"    Sign cond number:      {_demo_svs_sign[0]/_demo_svs_sign[-1]:.2f} (different structure entirely)")
print(f"    Only Muon FLATTENS the spectrum. The others merely RESCALE it.")

## Section 4: Learning Rate Sweep

Each optimizer has a different effective step size even at the same nominal learning rate, because
normalization changes the magnitude of the update vector. For a fair comparison, we sweep over
learning rate candidates independently for each optimizer and select the one that achieves the
lowest 200-step loss without divergence.

This is critical for scientific validity: we need to compare each optimizer **at its best**,
not at an arbitrary LR that might favor one over another. The stability threshold (loss < 50x initial)
prevents selecting unstable LRs that happen to produce a low snapshot but would eventually diverge.

Note that Sign SGD needs a much smaller LR because each element of the update is +/-1, giving
the step a Frobenius norm of $\sqrt{n^2} = n$, which is much larger than the unit-norm steps
of Normalized SGD or the unit-spectral-norm steps of Muon.

In [ ]:
def find_best_lr(method, net_type, num_steps=200):
    """
    Sweep over LR candidates for each method. Return the LR achieving the
    lowest final loss without diverging.
    """
    compute_loss_fn = compute_loss_linear if net_type == 'linear' else compute_loss_relu
    compute_grad_fn = compute_gradients_linear if net_type == 'linear' else compute_gradients_relu

    # Different LR ranges for different methods
    if method == 'sgd':
        candidates = [0.1, 0.07, 0.05, 0.03, 0.02, 0.015, 0.01, 0.007, 0.005, 0.003, 0.001]
    elif method == 'muon':
        candidates = [0.05, 0.03, 0.02, 0.015, 0.01, 0.007, 0.005, 0.003, 0.002, 0.001]
    elif method == 'norm_sgd':
        candidates = [0.05, 0.03, 0.02, 0.015, 0.01, 0.007, 0.005, 0.003, 0.002, 0.001, 0.0005]
    elif method == 'spectral_sgd':
        candidates = [0.05, 0.03, 0.02, 0.015, 0.01, 0.007, 0.005, 0.003, 0.002, 0.001, 0.0005]
    elif method == 'sign_sgd':
        # Sign SGD needs much smaller LR (each element is +/-1 times dim*dim)
        candidates = [0.005, 0.003, 0.002, 0.001, 0.0007, 0.0005, 0.0003, 0.0002, 0.0001, 0.00005]

    best_lr = candidates[-1]
    best_loss = float('inf')

    for lr_cand in candidates:
        np.random.seed(42)
        weights = init_weights(NUM_LAYERS)
        velocities = [np.zeros((DIM, DIM)) for _ in range(NUM_LAYERS)]
        initial_loss = compute_loss_fn(weights, X_data, W_target)
        stable = True
        final_loss = initial_loss

        for step in range(num_steps):
            grads = compute_grad_fn(weights, X_data, W_target)
            weights, velocities = optimizer_step(weights, velocities, grads, lr_cand, method)
            loss = compute_loss_fn(weights, X_data, W_target)
            if np.isnan(loss) or loss > initial_loss * 50:
                stable = False
                break
            final_loss = loss

        if stable and final_loss < best_loss:
            best_loss = final_loss
            best_lr = lr_cand

    return best_lr, best_loss

## Section 5: Training Engine

The training engine runs a single optimization trajectory from a given initialization. It records
the full loss curve and returns the final weights for downstream diversity analysis.

Divergence detection: if the loss becomes NaN or exceeds $10^{10}$, training is halted and the
remaining loss curve is padded with NaN. This prevents numerical overflow from corrupting the
convergence basin analysis.

In [ ]:
def run_training(weights_init, method, lr, num_steps, net_type):
    """
    Run optimizer for num_steps from weights_init.
    Returns loss curve and final weights.
    """
    compute_loss_fn = compute_loss_linear if net_type == 'linear' else compute_loss_relu
    compute_grad_fn = compute_gradients_linear if net_type == 'linear' else compute_gradients_relu

    weights = [w.copy() for w in weights_init]
    velocities = [np.zeros_like(w) for w in weights]
    losses = [compute_loss_fn(weights, X_data, W_target)]

    for step in range(num_steps):
        grads = compute_grad_fn(weights, X_data, W_target)
        weights, velocities = optimizer_step(weights, velocities, grads, lr, method)
        loss = compute_loss_fn(weights, X_data, W_target)
        losses.append(loss)
        if np.isnan(loss) or loss > 1e10:
            # Pad remaining with NaN
            for _ in range(num_steps - step - 1):
                losses.append(np.nan)
            break

    return np.array(losses), weights

## Section 6: Convergence Basin Analysis

This is the core measurement apparatus. For each optimizer, we run 20 independent training
trajectories from different random initializations (seeds 1000-1019). After training, we measure:

### Weight Diversity (d_w)
Mean pairwise Frobenius distance: $d_w = \frac{1}{\binom{20}{2}} \sum_{i < j} \sqrt{\sum_l \|W_l^{(i)} - W_l^{(j)}\|_F^2}$

This measures how spread out the final weight configurations are across different runs.
High weight diversity means the optimizer explores a wider region of weight space.

### Functional Diversity (d_f)
Mean pairwise output distance on held-out test data, normalized by data scale:
$d_f = \frac{1}{\binom{20}{2}} \sum_{i < j} \frac{\|f^{(i)}(X_\text{test}) - f^{(j)}(X_\text{test})\|_F}{\|X_\text{test}\|_F}$

This measures whether the different weight configurations actually produce different functions.
Low functional diversity means the optimizer converges to (approximately) the same input-output mapping.

### The Paradox Ratio (F/W)
$\text{Paradox Ratio} = d_f / d_w$

This is the key diagnostic. A low ratio means "weights differ a lot but functions agree" --
the hallmark of gauge exploration. The Muon paradox is the observation that Muon achieves
a much lower F/W ratio than SGD, meaning it explores the gauge orbit more aggressively.

### Condition Numbers
Per-layer condition number $\kappa(W_l) = \sigma_1(W_l) / \sigma_n(W_l)$ at convergence.
This reveals whether the optimizer drives weights toward well-conditioned or ill-conditioned configurations.

With $\binom{20}{2} = 190$ pairwise comparisons, we have strong statistics on diversity.

In [ ]:
def measure_convergence_basin(method, lr, net_type, num_runs=NUM_INDEPENDENT_RUNS,
                              num_steps=NUM_STEPS):
    """
    Run num_runs independent initializations. Measure:
      - Weight diversity: mean pairwise ||W_i - W_j||_F
      - Function diversity: mean pairwise ||f(X_test;W_i) - f(X_test;W_j)||_F
      - Loss mean and std
      - Per-layer condition number at convergence
    """
    forward_fn = forward_linear if net_type == 'linear' else forward_relu
    compute_loss_fn = compute_loss_linear if net_type == 'linear' else compute_loss_relu

    final_weights_list = []
    final_functions = []
    final_losses = []
    loss_curves = []
    condition_numbers = []  # per-layer condition numbers

    for run_idx in range(num_runs):
        weights_init = init_weights(NUM_LAYERS, seed=1000 + run_idx)
        loss_curve, final_weights = run_training(
            weights_init, method, lr, num_steps, net_type)

        loss_curves.append(loss_curve)
        final_weights_list.append(final_weights)
        final_functions.append(forward_fn(final_weights, X_test).copy())
        final_losses.append(compute_loss_fn(final_weights, X_data, W_target))

        # Per-layer condition number
        cond_per_layer = []
        for W in final_weights:
            svs = np.linalg.svd(W, compute_uv=False)
            if svs[-1] > 1e-15:
                cond_per_layer.append(svs[0] / svs[-1])
            else:
                cond_per_layer.append(np.inf)
        condition_numbers.append(cond_per_layer)

    # Pairwise diversity
    n = len(final_weights_list)
    weight_dists = []
    func_dists = []
    for i in range(n):
        for j in range(i + 1, n):
            d_w = 0.0
            for k in range(NUM_LAYERS):
                d_w += np.linalg.norm(
                    final_weights_list[i][k] - final_weights_list[j][k], 'fro') ** 2
            weight_dists.append(np.sqrt(d_w))
            d_f = np.linalg.norm(
                final_functions[i] - final_functions[j], 'fro') / np.linalg.norm(X_test, 'fro')
            func_dists.append(d_f)

    # Mean condition number per layer
    cond_arr = np.array(condition_numbers)  # shape (num_runs, NUM_LAYERS)
    mean_cond = np.mean(cond_arr, axis=0)

    # Mean loss curve (handle NaN)
    max_len = max(len(lc) for lc in loss_curves)
    padded = np.full((num_runs, max_len), np.nan)
    for i, lc in enumerate(loss_curves):
        padded[i, :len(lc)] = lc
    mean_loss_curve = np.nanmean(padded, axis=0)

    return {
        'weight_diversity_mean': np.mean(weight_dists),
        'weight_diversity_std': np.std(weight_dists),
        'func_diversity_mean': np.mean(func_dists),
        'func_diversity_std': np.std(func_dists),
        'loss_mean': np.mean(final_losses),
        'loss_std': np.std(final_losses),
        'losses': np.array(final_losses),
        'weight_dists': np.array(weight_dists),
        'func_dists': np.array(func_dists),
        'mean_loss_curve': mean_loss_curve,
        'mean_cond_per_layer': mean_cond,
        'cond_all': cond_arr,
    }

## Section 7: Main Experiment Execution

We now run the full experiment pipeline for both architectures (linear and ReLU):

1. **Phase 1 -- LR Sweep**: For each of the 5 optimizers, sweep over 10-11 LR candidates and select
   the best one based on 200-step stability and final loss.
2. **Phase 2 -- Convergence Basin**: Using the best LR, run 20 independent training trajectories
   for 500 steps each. Measure all diversity metrics.

This is the most compute-intensive part of the notebook: 5 optimizers x 2 architectures x
20 runs x 500 steps = 100,000 optimization steps total.

In [ ]:
print("=" * 110)
print("H3: NORMALIZED SGD vs MUON — Is the Polar Factor Necessary?")
print("=" * 110)
print(f"Setup: {NUM_LAYERS}-layer nets (dim={DIM}), quadratic loss, {NUM_STEPS} steps")
print(f"{NUM_INDEPENDENT_RUNS} independent inits per optimizer, momentum={MOMENTUM}")
print(f"Optimizers: SGD, Muon, Normalized SGD (Frob), Spectral-Norm SGD, Sign SGD")
print("=" * 110)

In [ ]:
all_results = {}

In [ ]:
for net_type in ['linear', 'relu']:
    print(f"\n{'#' * 110}")
    print(f"  ARCHITECTURE: {net_type.upper()} ({NUM_LAYERS}-layer, {DIM}x{DIM})")
    print(f"{'#' * 110}")

    # Phase 1: LR sweep for each optimizer
    print(f"\n  --- Phase 1: Learning Rate Sweep ---")
    print(f"  Sweeping {len(OPTIMIZER_NAMES)} optimizers over ~10 LR candidates each (200-step trials)...")
    best_lrs = {}
    for method in OPTIMIZER_NAMES:
        lr, loss = find_best_lr(method, net_type, num_steps=200)
        best_lrs[method] = lr
        print(f"    {OPTIMIZER_LABELS[method]:<30} best_lr={lr:.5f}  (200-step loss={loss:.6e})")

    print(f"\n  LR SWEEP INTERPRETATION:")
    print(f"    Observe the selected LRs -- they encode implicit information about each optimizer's")
    print(f"    effective step size. If Muon and Norm SGD select similar LRs, their update magnitudes")
    print(f"    are comparable. If Sign SGD selects a much smaller LR, it compensates for the larger")
    print(f"    Frobenius norm of the sign matrix (||sign(M)||_F = sqrt({DIM}*{DIM}) = {DIM:.0f}).")

    # Phase 2: Full training + convergence basin analysis (20 runs)
    print(f"\n  --- Phase 2: Convergence Basin ({NUM_INDEPENDENT_RUNS} runs, {NUM_STEPS} steps) ---")
    print(f"  Running {NUM_INDEPENDENT_RUNS} independent seeds per optimizer (190 pairwise comparisons)...")
    net_results = {}
    for method in OPTIMIZER_NAMES:
        t0 = time.time()
        result = measure_convergence_basin(method, best_lrs[method], net_type)
        elapsed = time.time() - t0
        net_results[method] = result
        net_results[method]['lr'] = best_lrs[method]
        print(f"    {OPTIMIZER_LABELS[method]:<30} loss={result['loss_mean']:.6e} +/- {result['loss_std']:.6e}  "
              f"d_w={result['weight_diversity_mean']:.4f}  d_f={result['func_diversity_mean']:.6f}  "
              f"({elapsed:.1f}s)")

    # Quick paradox ratio summary
    print(f"\n  QUICK PARADOX RATIO COMPARISON ({net_type.upper()}):")
    for method in OPTIMIZER_NAMES:
        r = net_results[method]
        _ratio = r['func_diversity_mean'] / r['weight_diversity_mean'] if r['weight_diversity_mean'] > 1e-15 else np.nan
        _bar = '#' * max(1, int(_ratio * 200))
        print(f"    {OPTIMIZER_LABELS[method]:<30} F/W = {_ratio:.6f}  {_bar}")

    all_results[net_type] = net_results

## Section 8: Detailed Results Tables

Three complementary views of the data:

1. **Convergence and Loss**: Which optimizers converge fastest and to what final loss?
   This tests whether normalization matches Muon's optimization speed.

2. **Paradox Metrics**: Weight diversity vs. functional diversity vs. loss consistency.
   This tests whether normalization creates the same gauge-exploration pattern as Muon.

3. **Per-Layer Condition Numbers**: The spectral health of converged weight matrices.
   This reveals whether the optimizer biases weights toward well-conditioned configurations.

Look for: Do the normalized variants cluster together? Does Muon stand apart? Does any
normalization scheme closely track Muon's numbers?

In [ ]:
for net_type in ['linear', 'relu']:
    res = all_results[net_type]

    print(f"\n\n{'=' * 110}")
    print(f"RESULTS TABLE: {net_type.upper()} NET")
    print(f"{'=' * 110}")

    # ---- Convergence & Loss ----
    print(f"\n  {'Optimizer':<30} | {'LR':>8} | {'Final Loss':>14} | {'Loss Std':>14} | {'Mean Loss Curve End':>18}")
    print(f"  {'-' * 100}")
    for m in OPTIMIZER_NAMES:
        r = res[m]
        curve_end = r['mean_loss_curve'][-1] if len(r['mean_loss_curve']) > 0 else np.nan
        print(f"  {OPTIMIZER_LABELS[m]:<30} | {r['lr']:>8.5f} | {r['loss_mean']:>14.6e} | "
              f"{r['loss_std']:>14.6e} | {curve_end:>18.6e}")

    # ---- Paradox Metrics ----
    print(f"\n  {'Optimizer':<30} | {'Weight Div':>12} | {'Func Div':>12} | {'F/W Ratio':>12} | {'Loss Std':>14}")
    print(f"  {'-' * 90}")
    _ratios_for_rank = {}
    for m in OPTIMIZER_NAMES:
        r = res[m]
        ratio = r['func_diversity_mean'] / r['weight_diversity_mean'] if r['weight_diversity_mean'] > 1e-15 else np.nan
        _ratios_for_rank[m] = ratio
        print(f"  {OPTIMIZER_LABELS[m]:<30} | {r['weight_diversity_mean']:>12.4f} | {r['func_diversity_mean']:>12.6f} | "
              f"{ratio:>12.6f} | {r['loss_std']:>14.6e}")

    # Rank optimizers by paradox strength
    _ranked = sorted(_ratios_for_rank.items(), key=lambda x: x[1])
    print(f"\n  PARADOX RANKING (lowest F/W ratio = strongest paradox):")
    for _rank, (_m, _r) in enumerate(_ranked, 1):
        print(f"    #{_rank}: {OPTIMIZER_LABELS[_m]:<30} F/W = {_r:.6f}")

    # ---- Per-Layer Condition Numbers ----
    print(f"\n  {'Optimizer':<30} |", end='')
    for l in range(NUM_LAYERS):
        print(f" {'Layer '+str(l):>10} |", end='')
    print(f" {'Geom Mean':>10}")
    print(f"  {'-' * (35 + 13 * NUM_LAYERS + 12)}")
    for m in OPTIMIZER_NAMES:
        r = res[m]
        conds = r['mean_cond_per_layer']
        geo_mean = np.exp(np.mean(np.log(np.clip(conds, 1e-15, None))))
        print(f"  {OPTIMIZER_LABELS[m]:<30} |", end='')
        for l in range(NUM_LAYERS):
            print(f" {conds[l]:>10.2f} |", end='')
        print(f" {geo_mean:>10.2f}")

    print(f"\n  CONDITIONING INTERPRETATION ({net_type.upper()}):")
    _sgd_cond = np.exp(np.mean(np.log(np.clip(res['sgd']['mean_cond_per_layer'], 1e-15, None))))
    _muon_cond = np.exp(np.mean(np.log(np.clip(res['muon']['mean_cond_per_layer'], 1e-15, None))))
    _norm_cond = np.exp(np.mean(np.log(np.clip(res['norm_sgd']['mean_cond_per_layer'], 1e-15, None))))
    print(f"    SGD geometric mean condition:      {_sgd_cond:.2f}")
    print(f"    Muon geometric mean condition:     {_muon_cond:.2f}")
    print(f"    Norm SGD geometric mean condition: {_norm_cond:.2f}")
    if _muon_cond < _sgd_cond * 0.8:
        print(f"    Muon drives weights toward BETTER conditioning than SGD.")
        print(f"    This is consistent with the polar factor biasing updates toward orthogonal structure.")
    elif _muon_cond > _sgd_cond * 1.2:
        print(f"    Muon allows HIGHER condition numbers than SGD -- gauge exploration is real.")
    else:
        print(f"    Muon and SGD produce similar conditioning.")

## Section 9: Key Hypothesis Tests

We now apply four formal tests to discriminate between Hypothesis A (scale removal suffices)
and Hypothesis B (polar factor is necessary):

**T1 -- Paradox Creation:** Does Normalized SGD produce a lower F/W ratio than SGD?
If yes, then *some* form of normalization creates the paradox. This is a necessary condition
for Hypothesis A but not sufficient (normalization might create a weaker paradox than Muon).

**T2 -- Convergence Speed Match:** Does Normalized SGD match Muon's convergence speed?
Measured at both 50% and 100% of training. "Match" is defined as loss ratios within 3x.
If no match, the polar factor provides convergence benefits beyond scale removal.

**T3 -- Loss Quality:** Does Muon achieve a lower final loss than Normalized SGD?
This directly tests whether the orthogonal projection provides gradient direction quality
that simple normalization cannot.

**T4 -- Paradox Strength Ranking:** Among all normalization variants, does Muon have the
strongest paradox (lowest F/W ratio)? If Muon is uniquely strongest, the polar factor does
something qualitatively different from any normalization scheme.

In [ ]:
print(f"\n\n{'=' * 110}")
print("KEY HYPOTHESIS TESTS")
print("=" * 110)

In [ ]:
total_pass = 0
total_tests = 0
all_test_results = {}

In [ ]:
for net_type in ['linear', 'relu']:
    res = all_results[net_type]
    print(f"\n  {'=' * 80}")
    print(f"  {net_type.upper()} NET")
    print(f"  {'=' * 80}")

    # Helper: compute paradox ratio for a method
    def paradox_ratio(m):
        r = res[m]
        if r['weight_diversity_mean'] > 1e-15:
            return r['func_diversity_mean'] / r['weight_diversity_mean']
        return np.nan

    # Helper: paradox strength = weight_div * (1 / func_div) * (1 / loss_std)
    # Higher = stronger paradox (more weight diversity, less function diversity, less loss spread)
    def paradox_strength(m):
        r = res[m]
        if r['func_diversity_mean'] > 1e-15 and r['loss_std'] > 1e-20:
            return r['weight_diversity_mean'] / (r['func_diversity_mean'] * r['loss_std'])
        return 0.0

    # ----- T1: Does normalized SGD create the paradox? -----
    # Paradox = high weight diversity AND low loss std AND low func diversity
    # Compare to SGD baseline
    sgd_ratio = paradox_ratio('sgd')
    norm_ratio = paradox_ratio('norm_sgd')
    muon_ratio = paradox_ratio('muon')

    t1 = norm_ratio < sgd_ratio  # Lower ratio means stronger paradox
    total_tests += 1
    total_pass += t1
    print(f"\n  T1: Does Normalized SGD create the paradox?")
    print(f"      F/W Ratio: SGD={sgd_ratio:.6f}, Norm_SGD={norm_ratio:.6f}, Muon={muon_ratio:.6f}")
    print(f"      Norm SGD ratio < SGD ratio? {norm_ratio:.6f} < {sgd_ratio:.6f} --> {'PASS' if t1 else 'FAIL'}")

    # ----- T2: Does normalized SGD match Muon convergence speed? -----
    # Compare mean loss curves at 50%, 100% of steps
    muon_curve = res['muon']['mean_loss_curve']
    norm_curve = res['norm_sgd']['mean_loss_curve']
    half = len(muon_curve) // 2
    # Match = within 2x of each other at 50% and 100%
    muon_half = muon_curve[half] if half < len(muon_curve) else np.nan
    norm_half = norm_curve[half] if half < len(norm_curve) else np.nan
    muon_final = res['muon']['loss_mean']
    norm_final = res['norm_sgd']['loss_mean']
    # Within 3x is a rough match
    t2_half = (min(muon_half, norm_half) / max(muon_half, norm_half) > 0.33) if (muon_half > 1e-15 and norm_half > 1e-15) else False
    t2_final = (min(muon_final, norm_final) / max(muon_final, norm_final) > 0.33) if (muon_final > 1e-15 and norm_final > 1e-15) else False
    t2 = t2_half and t2_final
    total_tests += 1
    total_pass += t2
    print(f"\n  T2: Does Normalized SGD match Muon convergence speed?")
    print(f"      At 50% steps: Muon={muon_half:.6e}, Norm={norm_half:.6e}, ratio={min(muon_half,norm_half)/max(muon_half,norm_half):.3f}")
    print(f"      Final loss:   Muon={muon_final:.6e}, Norm={norm_final:.6e}, ratio={min(muon_final,norm_final)/max(muon_final,norm_final):.3f}")
    print(f"      --> {'PASS (comparable)' if t2 else 'FAIL (not comparable)'}")

    # ----- T3: Does Muon beat normalized SGD on LOSS? -----
    t3 = muon_final < norm_final
    total_tests += 1
    total_pass += t3
    print(f"\n  T3: Does Muon beat Normalized SGD on final loss?")
    print(f"      Muon={muon_final:.6e} vs Norm={norm_final:.6e}")
    print(f"      --> {'PASS (Muon wins)' if t3 else 'FAIL (Norm wins or tie)'}")

    # ----- T4: Which normalization creates strongest paradox? -----
    norm_methods = ['norm_sgd', 'spectral_sgd', 'sign_sgd', 'muon']
    ratios = {m: paradox_ratio(m) for m in norm_methods}
    strengths = {m: paradox_strength(m) for m in norm_methods}
    best_ratio = min(ratios, key=lambda m: ratios[m])
    best_strength = max(strengths, key=lambda m: strengths[m])
    total_tests += 1
    t4 = best_ratio == 'muon'
    total_pass += t4
    print(f"\n  T4: Which normalization creates strongest paradox?")
    print(f"      F/W Ratios (lower = stronger paradox):")
    for m in norm_methods:
        marker = " <-- BEST" if m == best_ratio else ""
        print(f"        {OPTIMIZER_LABELS[m]:<30} ratio={ratios[m]:.6f}{marker}")
    print(f"      Paradox Strength (weight_div / (func_div * loss_std)):")
    for m in norm_methods:
        marker = " <-- BEST" if m == best_strength else ""
        print(f"        {OPTIMIZER_LABELS[m]:<30} strength={strengths[m]:.2f}{marker}")
    print(f"      Muon has lowest F/W ratio? --> {'PASS' if t4 else 'FAIL'}")

    all_test_results[net_type] = {
        't1': t1, 't2': t2, 't3': t3, 't4': t4,
        'ratios': ratios, 'strengths': strengths,
    }

## Section 10: The Critical Comparison -- Is the Polar Factor Necessary?

This section synthesizes the test results into a coherent scientific conclusion. We check
whether any simple normalization scheme (Frobenius, spectral, or sign) can match Muon on
**both** paradox strength and loss quality simultaneously.

The logic tree:

- If normalization matches on **both** dimensions: Polar factor is unnecessary (Hypothesis A wins)
- If normalization matches on paradox **but not loss**: Two-mechanism story -- scale removal creates
  gauge exploration, but the polar factor additionally provides superior gradient direction quality
- If normalization matches on **neither**: Polar factor is doing something qualitatively different
  (Hypothesis B wins)

The thresholds are deliberately generous: "matches paradox" requires F/W ratio at least 20% better
than SGD, "matches loss" requires final loss within 50% of Muon. These favor Hypothesis A --
if even with generous thresholds normalization fails to match, the conclusion is stronger.

In [ ]:
print(f"\n\n{'=' * 110}")
print("THE CRITICAL COMPARISON: Is the Polar Factor Necessary?")
print("=" * 110)

In [ ]:
for net_type in ['linear', 'relu']:
    res = all_results[net_type]
    tr = all_test_results[net_type]

    print(f"\n  --- {net_type.upper()} NET ---")

    muon_loss = res['muon']['loss_mean']
    norm_loss = res['norm_sgd']['loss_mean']
    spec_loss = res['spectral_sgd']['loss_mean']

    muon_ratio = tr['ratios']['muon']
    norm_ratio = tr['ratios']['norm_sgd']
    spec_ratio = tr['ratios']['spectral_sgd']
    sgd_ratio = res['sgd']['func_diversity_mean'] / res['sgd']['weight_diversity_mean'] if res['sgd']['weight_diversity_mean'] > 1e-15 else np.nan

    # Check if norm/spectral SGD match Muon on BOTH paradox AND loss
    norm_matches_paradox = norm_ratio < sgd_ratio * 0.8  # At least 20% better than SGD
    norm_matches_loss = abs(norm_loss - muon_loss) / max(muon_loss, 1e-15) < 0.5  # Within 50%
    spec_matches_paradox = spec_ratio < sgd_ratio * 0.8
    spec_matches_loss = abs(spec_loss - muon_loss) / max(muon_loss, 1e-15) < 0.5

    any_matches_both = (norm_matches_paradox and norm_matches_loss) or (spec_matches_paradox and spec_matches_loss)
    any_matches_paradox_only = (norm_matches_paradox or spec_matches_paradox) and not any_matches_both

    print(f"\n    SGD baseline F/W ratio:        {sgd_ratio:.6f}")
    print(f"    Muon F/W ratio:                {muon_ratio:.6f}  loss={muon_loss:.6e}")
    print(f"    Norm SGD F/W ratio:            {norm_ratio:.6f}  loss={norm_loss:.6e}")
    print(f"    Spectral SGD F/W ratio:        {spec_ratio:.6f}  loss={spec_loss:.6e}")
    print(f"    Sign SGD F/W ratio:            {tr['ratios']['sign_sgd']:.6f}  loss={res['sign_sgd']['loss_mean']:.6e}")

    print(f"\n    Norm SGD paradox?  {norm_matches_paradox}  (ratio < 80% of SGD)")
    print(f"    Norm SGD loss match?  {norm_matches_loss}  (within 50% of Muon)")
    print(f"    Spec SGD paradox?  {spec_matches_paradox}")
    print(f"    Spec SGD loss match?  {spec_matches_loss}")

    if any_matches_both:
        print(f"\n    ==> CONCLUSION: Polar factor is UNNECESSARY.")
        print(f"        Simple normalization suffices for both paradox and convergence.")
        print(f"        The key mechanism is scale removal, not orthogonal projection.")
    elif any_matches_paradox_only or (norm_matches_paradox and not norm_matches_loss) or (spec_matches_paradox and not spec_matches_loss):
        print(f"\n    ==> CONCLUSION: Normalization creates the paradox but DIRECTIONAL QUALITY matters.")
        print(f"        The polar factor provides better convergence than simple normalization.")
        print(f"        Scale removal creates gauge exploration; ortho projection provides gradient quality.")
    else:
        print(f"\n    ==> CONCLUSION: Polar factor is doing something QUALITATIVELY DIFFERENT.")
        print(f"        Simple normalization does NOT replicate the Muon paradox or convergence.")
        print(f"        The orthogonal projection is essential, not just the normalization.")

## Section 11: Comprehensive Numbers for Reference

A single table combining all metrics for both architectures, suitable for inclusion in
a paper or for cross-referencing with other experiments in the H3 series (H3a, H3b).

In [ ]:
print(f"\n\n{'=' * 110}")
print("COMPREHENSIVE NUMBERS (for paper)")
print("=" * 110)

In [ ]:
for net_type in ['linear', 'relu']:
    res = all_results[net_type]
    print(f"\n  {net_type.upper()} NET:")
    print(f"  {'Method':<30} | {'Loss':>12} | {'Loss Std':>12} | {'W Div':>10} | {'F Div':>12} | "
          f"{'F/W Ratio':>10} | {'Cond(geom)':>10} | {'Paradox Str':>12}")
    print(f"  {'-' * 125}")
    for m in OPTIMIZER_NAMES:
        r = res[m]
        ratio = r['func_diversity_mean'] / r['weight_diversity_mean'] if r['weight_diversity_mean'] > 1e-15 else np.nan
        conds = r['mean_cond_per_layer']
        geo_cond = np.exp(np.mean(np.log(np.clip(conds, 1e-15, None))))
        if r['func_diversity_mean'] > 1e-15 and r['loss_std'] > 1e-20:
            pstr = r['weight_diversity_mean'] / (r['func_diversity_mean'] * r['loss_std'])
        else:
            pstr = 0.0
        print(f"  {OPTIMIZER_LABELS[m]:<30} | {r['loss_mean']:>12.6e} | {r['loss_std']:>12.6e} | "
              f"{r['weight_diversity_mean']:>10.4f} | {r['func_diversity_mean']:>12.6f} | "
              f"{ratio:>10.6f} | {geo_cond:>10.2f} | {pstr:>12.2f}")

## Section 12: Visualization

Six panels per architecture, plus a combined summary:

- **(a) Loss Curves**: Mean loss over 20 runs for each optimizer. Look for: which optimizers
  converge fastest? Does Muon's curve separate from the normalization variants?
- **(b) Final Loss Bar Chart**: End-state comparison with error bars. Log scale.
- **(c) Paradox Ratio**: The key diagnostic -- F/W ratio for each optimizer. Lower = stronger paradox.
- **(d) Weight Diversity**: How spread out are the final weights across runs?
- **(e) Functional Diversity**: How different are the actual outputs? (Should be low for all good optimizers)
- **(f) Condition Numbers**: Per-layer spectral health of converged weights.

In [ ]:
print(f"\n\n{'=' * 110}")
print("GENERATING PLOTS")
print("=" * 110)

In [ ]:
try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    for net_type in ['linear', 'relu']:
        res = all_results[net_type]

        fig, axes = plt.subplots(2, 3, figsize=(22, 14))
        fig.suptitle(
            f'H3: Normalized SGD vs Muon ({net_type.upper()} net)\n'
            f'{NUM_LAYERS}-layer, dim={DIM}, {NUM_STEPS} steps, {NUM_INDEPENDENT_RUNS} runs',
            fontsize=14, fontweight='bold')

        # ---- (a) Loss Curves ----
        ax = axes[0, 0]
        ax.set_title('Mean Loss Curves (Best LR)')
        for m in OPTIMIZER_NAMES:
            curve = res[m]['mean_loss_curve']
            ax.semilogy(np.arange(len(curve)), curve, color=OPTIMIZER_COLORS[m],
                        linewidth=2, label=f"{OPTIMIZER_LABELS[m]} (lr={res[m]['lr']:.4f})")
        ax.set_xlabel('Step')
        ax.set_ylabel('Loss')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

        # ---- (b) Final Loss Bar Chart ----
        ax = axes[0, 1]
        ax.set_title('Final Loss (mean +/- std)')
        x_pos = np.arange(len(OPTIMIZER_NAMES))
        means = [res[m]['loss_mean'] for m in OPTIMIZER_NAMES]
        stds = [res[m]['loss_std'] for m in OPTIMIZER_NAMES]
        colors = [OPTIMIZER_COLORS[m] for m in OPTIMIZER_NAMES]
        bars = ax.bar(x_pos, means, yerr=stds, color=colors, edgecolor='black',
                      capsize=3, width=0.6)
        ax.set_xticks(x_pos)
        ax.set_xticklabels([OPTIMIZER_LABELS[m] for m in OPTIMIZER_NAMES],
                           rotation=30, ha='right', fontsize=8)
        ax.set_ylabel('Final Loss')
        ax.set_yscale('log')
        ax.grid(True, alpha=0.3, axis='y')
        for i, bar in enumerate(bars):
            ax.text(bar.get_x() + bar.get_width() / 2., bar.get_height(),
                    f'{means[i]:.2e}', ha='center', va='bottom', fontsize=7, fontweight='bold')

        # ---- (c) Paradox Ratio (F/W) ----
        ax = axes[0, 2]
        ax.set_title('Paradox Ratio: Func_Div / Weight_Div\n(Lower = stronger paradox)')
        ratios = []
        for m in OPTIMIZER_NAMES:
            r = res[m]
            if r['weight_diversity_mean'] > 1e-15:
                ratios.append(r['func_diversity_mean'] / r['weight_diversity_mean'])
            else:
                ratios.append(0)
        bars = ax.bar(x_pos, ratios, color=colors, edgecolor='black', width=0.6)
        ax.set_xticks(x_pos)
        ax.set_xticklabels([OPTIMIZER_LABELS[m] for m in OPTIMIZER_NAMES],
                           rotation=30, ha='right', fontsize=8)
        ax.set_ylabel('Func Div / Weight Div')
        ax.grid(True, alpha=0.3, axis='y')
        for i, bar in enumerate(bars):
            ax.text(bar.get_x() + bar.get_width() / 2., bar.get_height(),
                    f'{ratios[i]:.4f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

        # ---- (d) Weight Diversity ----
        ax = axes[1, 0]
        ax.set_title('Weight Diversity (pairwise)')
        wdivs = [res[m]['weight_diversity_mean'] for m in OPTIMIZER_NAMES]
        bars = ax.bar(x_pos, wdivs, color=colors, edgecolor='black', width=0.6)
        ax.set_xticks(x_pos)
        ax.set_xticklabels([OPTIMIZER_LABELS[m] for m in OPTIMIZER_NAMES],
                           rotation=30, ha='right', fontsize=8)
        ax.set_ylabel('Mean Pairwise ||W_i - W_j||_F')
        ax.grid(True, alpha=0.3, axis='y')
        for i, bar in enumerate(bars):
            ax.text(bar.get_x() + bar.get_width() / 2., bar.get_height(),
                    f'{wdivs[i]:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

        # ---- (e) Function Diversity ----
        ax = axes[1, 1]
        ax.set_title('Function Diversity (pairwise)')
        fdivs = [res[m]['func_diversity_mean'] for m in OPTIMIZER_NAMES]
        bars = ax.bar(x_pos, fdivs, color=colors, edgecolor='black', width=0.6)
        ax.set_xticks(x_pos)
        ax.set_xticklabels([OPTIMIZER_LABELS[m] for m in OPTIMIZER_NAMES],
                           rotation=30, ha='right', fontsize=8)
        ax.set_ylabel('Mean Pairwise ||f_i - f_j||_F / ||X||_F')
        ax.grid(True, alpha=0.3, axis='y')
        for i, bar in enumerate(bars):
            ax.text(bar.get_x() + bar.get_width() / 2., bar.get_height(),
                    f'{fdivs[i]:.5f}', ha='center', va='bottom', fontsize=7, fontweight='bold')

        # ---- (f) Condition Number ----
        ax = axes[1, 2]
        ax.set_title('Per-Layer Condition Number (mean)')
        x_layers = np.arange(NUM_LAYERS)
        width_bar = 0.15
        for idx, m in enumerate(OPTIMIZER_NAMES):
            conds = res[m]['mean_cond_per_layer']
            offset = (idx - len(OPTIMIZER_NAMES) / 2 + 0.5) * width_bar
            ax.bar(x_layers + offset, conds, width_bar, color=OPTIMIZER_COLORS[m],
                   edgecolor='black', label=OPTIMIZER_LABELS[m])
        ax.set_xticks(x_layers)
        ax.set_xticklabels([f'Layer {l}' for l in range(NUM_LAYERS)])
        ax.set_ylabel('Condition Number')
        ax.legend(fontsize=7, loc='upper left')
        ax.grid(True, alpha=0.3, axis='y')

        plt.tight_layout()
        plot_path = os.path.join(SCRIPT_DIR, f'h3_normalized_sgd_vs_muon_{net_type}.png')
        plt.savefig(plot_path, dpi=150, bbox_inches='tight')
        plt.close()
        print(f"  Plot saved: {plot_path}")

    # ---- Combined summary ----
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    fig.suptitle('H3: Is the Polar Factor Necessary?\n'
                 'Paradox Ratio (F_div / W_div) — Lower = Stronger Paradox',
                 fontsize=14, fontweight='bold')

    for idx, net_type in enumerate(['linear', 'relu']):
        res = all_results[net_type]
        ax = axes[idx]
        ax.set_title(f'{net_type.upper()} Net')

        ratios = []
        for m in OPTIMIZER_NAMES:
            r = res[m]
            if r['weight_diversity_mean'] > 1e-15:
                ratios.append(r['func_diversity_mean'] / r['weight_diversity_mean'])
            else:
                ratios.append(0)

        x_pos = np.arange(len(OPTIMIZER_NAMES))
        colors = [OPTIMIZER_COLORS[m] for m in OPTIMIZER_NAMES]
        bars = ax.bar(x_pos, ratios, color=colors, edgecolor='black', width=0.6)
        ax.set_xticks(x_pos)
        ax.set_xticklabels([OPTIMIZER_LABELS[m] for m in OPTIMIZER_NAMES],
                           rotation=25, ha='right', fontsize=9)
        ax.set_ylabel('Func Diversity / Weight Diversity')
        ax.grid(True, alpha=0.3, axis='y')
        for i, bar in enumerate(bars):
            ax.text(bar.get_x() + bar.get_width() / 2., bar.get_height(),
                    f'{ratios[i]:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

        # Add loss info as text
        loss_text = "Final Losses:\n"
        for m in OPTIMIZER_NAMES:
            loss_text += f"  {OPTIMIZER_LABELS[m]}: {res[m]['loss_mean']:.2e}\n"
        ax.text(0.98, 0.98, loss_text, transform=ax.transAxes, fontsize=7,
                ha='right', va='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    plt.tight_layout()
    combined_path = os.path.join(SCRIPT_DIR, 'h3_combined_summary.png')
    plt.savefig(combined_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Combined plot saved: {combined_path}")

In [ ]:
except ImportError:
    print("  WARNING: matplotlib not available, skipping plots.")

## Section 13: Final Verdict and Conclusions

### Test Summary

Below we tally the pass/fail results across both architectures. The pattern of passes and failures
determines which hypothesis the data supports:

| Pattern | Interpretation |
|---------|---------------|
| T1=PASS, T2=PASS across both | Hypothesis A: scale removal suffices |
| T1=PASS, T3=PASS across both | Two-mechanism: scale for paradox, polar for convergence |
| T4=PASS across both | Hypothesis B: polar factor is uniquely necessary |
| Mixed results | More nuanced; see per-architecture breakdown |

In [ ]:
print(f"\n\n{'=' * 110}")
print("FINAL VERDICT")
print("=" * 110)

In [ ]:
for net_type in ['linear', 'relu']:
    tr = all_test_results[net_type]
    res = all_results[net_type]
    print(f"\n  {net_type.upper()} NET:")
    tests = [('T1', tr['t1'], 'Norm SGD creates paradox (lower F/W ratio than SGD)'),
             ('T2', tr['t2'], 'Norm SGD matches Muon convergence speed'),
             ('T3', tr['t3'], 'Muon beats Norm SGD on final loss'),
             ('T4', tr['t4'], 'Muon has strongest paradox (lowest F/W ratio)')]
    for tname, tval, desc in tests:
        print(f"    {tname}: {'PASS' if tval else 'FAIL'}  -- {desc}")
    n_pass = sum(1 for _, v, _ in tests if v)
    print(f"    Total: {n_pass}/4 tests passed")

In [ ]:
all_pass = sum(1 for net_type in ['linear', 'relu']
               for v in [all_test_results[net_type]['t1'], all_test_results[net_type]['t2'],
                         all_test_results[net_type]['t3'], all_test_results[net_type]['t4']] if v)

In [ ]:
print(f"\n  Overall: {all_pass}/8 tests passed across both architectures")

In [ ]:
# Determine the paper thesis
print(f"\n  {'=' * 80}")
print(f"  PAPER THESIS DETERMINATION:")
print(f"  {'=' * 80}")

In [ ]:
# Check the dominant pattern
both_paradox = all(all_test_results[nt]['t1'] for nt in ['linear', 'relu'])
both_conv = all(all_test_results[nt]['t2'] for nt in ['linear', 'relu'])
muon_wins_loss = all(all_test_results[nt]['t3'] for nt in ['linear', 'relu'])
muon_wins_paradox = all(all_test_results[nt]['t4'] for nt in ['linear', 'relu'])

In [ ]:
if both_paradox and both_conv:
    print(f"  Normalization alone creates the paradox AND matches Muon convergence.")
    print(f"  ==> The polar factor is NOT necessary. Scale removal is the key mechanism.")
elif both_paradox and muon_wins_loss:
    print(f"  Normalization creates the paradox BUT Muon wins on loss.")
    print(f"  ==> The paradox comes from scale removal, but the polar factor provides")
    print(f"      superior gradient DIRECTION quality for faster convergence.")
    print(f"  ==> TWO-MECHANISM STORY: scale removal for gauge exploration,")
    print(f"      orthogonal projection for convergence quality.")
elif both_paradox:
    print(f"  Normalization creates the paradox. Mixed results on convergence.")
    print(f"  ==> Scale removal is sufficient for the paradox mechanism.")
elif muon_wins_paradox:
    print(f"  Only Muon creates strong paradox. The polar factor is essential.")
    print(f"  ==> The orthogonal projection does something qualitatively different")
    print(f"      from simple normalization.")
else:
    print(f"  Mixed results. The picture is more nuanced than any simple thesis.")

In [ ]:
print(f"\n{'=' * 110}")
print("EXPERIMENT COMPLETE")
print(f"{'=' * 110}")

## Conclusions and Implications for the RG-Gauge-Fixing Theory

### What We Tested
This experiment directly addressed whether Muon's advantage arises from a trivial mechanism
(removing gradient scale) or a non-trivial one (orthogonal projection onto the Stiefel manifold).
We compared five optimizers spanning the spectrum from no normalization (SGD) to full spectral
flattening (Muon), with three intermediate normalization schemes.

### Key Findings

**Finding 1 -- The Normalization Spectrum:** The five optimizers form a hierarchy of how aggressively
they modify the gradient's spectral structure:
- SGD: no modification (preserves the full spectrum)
- Spectral SGD: caps the largest singular value (partial spectral control)
- Frobenius SGD: normalizes total energy (global scale control)
- Sign SGD: destroys all magnitude information element-wise (aggressive but unstructured)
- Muon: replaces the entire spectrum with flat unit values (structured spectral equalization)

**Finding 2 -- Paradox vs. Convergence Decomposition:** The paradox ratio (F/W) and the final loss
can decouple. This suggests that the mechanisms behind gauge exploration (creating weight diversity
without functional diversity) and optimization quality (reaching low loss) may be partially independent.

**Finding 3 -- The Polar Factor's Unique Property:** Only the polar factor simultaneously achieves
spectral equalization AND preserves the gradient's directional structure (the left and right singular
vectors $U$ and $V$). Frobenius normalization preserves directions but retains spectral anisotropy.
Sign SGD destroys both spectral and directional structure. This places the polar factor in a unique
position: maximally aggressive on scale, maximally conservative on direction.

### Implications for the RG Framework

If the data shows that normalization creates the paradox but not the convergence benefit, this
supports a **two-layer RG interpretation**:

1. **Scale gauge fixing** (any normalization): Removes the trivial scale redundancy in the gauge
   group, enabling exploration of the gauge orbit. This is the "easy" part of gauge fixing.
2. **Rotational gauge fixing** (polar factor specifically): Projects updates onto the orthogonal
   group $O(n)$, which is the *structure-preserving* subgroup of the full gauge group $GL(n)$.
   This provides superior gradient direction by distributing update energy equally across all
   singular directions, preventing the optimizer from being trapped by spectral imbalance.

### Connections to Other Experiments
- **H3a** (Per-SV Gradient Utilization): examines *which* singular directions benefit most
- **H3b** (Partial SV Equalization): tests intermediate levels of spectral flattening
- **H6** (LR Artifact Check): verifies the results are not confounded by LR selection
- **H15** (Normalization-Gauge Duality): deeper theoretical analysis of the gauge structure